# 03 — From counts to cell types

**Day 1, 12:00–12:30 and Day 2, 09:30–10:15**

### Where we are going

The pipeline looks identical to scanpy — normalise, PCA, neighbours, Leiden,
markers. Three things are genuinely different, and if you carry the scRNA-seq
habits across unexamined you will get a plausible, wrong answer.

1. **There is no highly-variable-gene step.** The panel already made that choice.
2. **Normalisation has an extra option scRNA-seq does not have: cell area.**
3. **You can validate the clustering by looking at the tissue** — an external check
   that a UMAP can never give you.

By the end you can normalise a targeted panel defensibly, cluster it, annotate it
with markers, recognise the clusters that are segmentation artefacts rather than
cell types, and map the whole thing back onto the section.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=90, frameon=False)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

NAVY, GOLD, CORAL, ICE = "#001158", "#FBAE40", "#F26B43", "#BCD2FF"

In [ ]:
adata = sc.read_h5ad(DATA / "ovarian_qc.h5ad")   # from notebook 02
adata

## 1. Normalisation

Cells differ in total counts partly because they are more active, partly because
someone drew a bigger polygon around them, and partly because that corner of the
section yielded more signal. We use the standard approach — scale each cell to the
median total, then `log1p`:

```python
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
```

**One caveat, stated once.** This does not fully remove sequencing depth, and on a
targeted panel at a few hundred counts per cell the residue is larger than you are
used to from scRNA-seq. You will see it directly in section 2: the first principal
component of this dataset correlates strongly with the number of genes detected per
cell. It is a real property of the data, not a mistake in the pipeline.

Alternatives exist — dividing by cell area, Pearson residuals, regressing depth out —
and each fixes some artefacts while creating others. Appendix B
(`A2_normalisation_and_depth.ipynb`) works through all of them with a diagnostic that
says which is appropriate for a given dataset. For today, median scaling is the
default you will meet in other people's code, so it is the one to learn.

> **No highly-variable-gene step.** In scRNA-seq you pick ~2,000 variable genes out of
> 20,000 to denoise the distance metric. Here the panel is already a curated 5,000
> genes chosen to discriminate cell types. Running `highly_variable_genes` on top
> mostly selects the most *abundant* genes and discards the rare-but-informative
> markers the panel was designed around. Skip it.

In [ ]:
adata.layers["counts"] = adata.X.copy()      # keep raw counts safe, always

sc.pp.normalize_total(adata)                 # scale every cell to the median total
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

print(f"median total counts before: {np.median(np.asarray(adata.layers['counts'].sum(axis=1))):.0f}")
print("layers now available:", list(adata.layers.keys()))

## 2. Dimensionality reduction and clustering

Three parameters shape every cluster you are about to see. None has a correct value,
and the point of this section is that you develop a feel for what each one does.

| Parameter | Controls | Typical |
|---|---|---|
| `n_pcs` | how many principal components feed the graph | 20–50 for a 5K panel |
| `n_neighbors` | how local the expression graph is | 10–30 |
| `resolution` | how finely Leiden cuts that graph | 0.3–2.0 |

We fit the PCA once, then explore on a subsample so each run takes seconds rather than
minutes. When you have settled on values, the last cell applies them to everything.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, n_comps=50, svd_solver="arpack")
adata.X = adata.layers["lognorm"].copy()     # scaling was for PCA only
adata.uns["log1p"] = {"base": None}

vr = adata.uns["pca"]["variance_ratio"]
print(f"PC1 explains {100 * vr[0]:.2f}% of the variance; "
      f"the first 30 PCs together {100 * vr[:30].sum():.1f}%")
print("A flat spectrum like this is normal for a targeted panel at a few hundred")
print("counts per cell — no single axis dominates, so you need more components.\n")

# The caveat from section 1, made concrete. Depth lives in the early PCs.
for i in range(3):
    r_counts = np.corrcoef(adata.obsm["X_pca"][:, i], adata.obs["total_counts"])[0, 1]
    r_genes = np.corrcoef(adata.obsm["X_pca"][:, i], adata.obs["n_genes_by_counts"])[0, 1]
    r_area = np.corrcoef(adata.obsm["X_pca"][:, i], adata.obs["cell_area"])[0, 1]
    print(f"  PC{i + 1}: r with counts {r_counts:+.2f}, genes detected {r_genes:+.2f}, "
          f"area {r_area:+.2f}")
print("\nIf PC1 tracks counts and genes but not area, that is a detection-rate")
print("axis. Appendix B is about what to do; for now, be aware it is there.")

### Explore, on a subsample

Each run below takes a few seconds. Change one parameter at a time and watch what
happens to the number of clusters and to the UMAP.

In [ ]:
# A fast playground: same pipeline, fewer cells.
N_EXPLORE = 8000
rng = np.random.default_rng(0)
idx = rng.choice(adata.n_obs, size=min(N_EXPLORE, adata.n_obs), replace=False)
sub = adata[idx].copy()
print(f"exploring on {sub.n_obs:,} cells")


def explore(n_pcs=30, n_neighbors=15, resolution=1.0,
            min_dist=0.5, spread=1.0, show=True):
    """Neighbours -> UMAP -> Leiden with the given parameters, then plot."""
    sc.pp.neighbors(sub, n_neighbors=n_neighbors, n_pcs=n_pcs)
    sc.tl.umap(sub, min_dist=min_dist, spread=spread)
    sc.tl.leiden(sub, resolution=resolution, key_added="cl",
                 flavor="igraph", n_iterations=2)
    n = sub.obs["cl"].nunique()
    sizes = sub.obs["cl"].value_counts()
    print(f"n_pcs={n_pcs}  n_neighbors={n_neighbors}  resolution={resolution}  "
          f"min_dist={min_dist}  ->  {n} clusters, "
          f"smallest {sizes.iloc[-1]:,} cells")
    if show:
        sc.pl.umap(sub, color=["cl", "total_counts"], wspace=0.3, size=12,
                   title=[f"{n} clusters", "total_counts"])
    return n


explore()          # the defaults

> **Try it yourself — how many PCs?**
>
> Too few and rare cell types collapse into their neighbours; too many adds noise,
> though Leiden tolerates that better than you would expect. Watch the smallest
> cluster: if it disappears, you cut too deep.

In [ ]:
explore(n_pcs=10)     # <-- CHANGE THIS (try 10, 20, 30, 50)

> **Try it yourself — how local is the graph?**
>
> `n_neighbors` decides how many cells each cell is connected to. Small values find
> fine structure and more clusters, at the cost of noise; large values smooth
> everything and merge subtypes. Note this is the **expression** graph — nothing to do
> with the spatial neighbours you will build tomorrow.

In [ ]:
explore(n_neighbors=5)     # <-- CHANGE THIS (try 5, 15, 30, 50)

> **Try it yourself — how finely to cut?**
>
> `resolution` is not a number of clusters: doubling it does not double them. Sweep it
> and pick by a question you can answer — can you name every cluster with markers?

### A second detour: the `for` loop

Running `explore()` four times by hand would work, but nobody does that. This is the
other piece of Python syntax worth understanding, and it has exactly four parts:

```python
for r in [0.3, 0.6, 1.0, 2.0]:
    explore(resolution=r, show=False)
```

1. **`for`** — start a loop.
2. **`r`** — a name *you* invent. It takes each value in turn: first `r` is 0.3, then
   0.6, and so on. Call it `res` or `x` if you prefer; nothing depends on the name.
3. **`in [0.3, 0.6, 1.0, 2.0]`** — the list to walk through. Square brackets make a
   list in Python.
4. **`:` then an indented line** — the colon ends the header, and everything indented
   below it is the body, run once per value.

**Indentation is not decoration.** In Python it *is* the syntax — it defines what
belongs inside the loop. Four spaces is the convention, and Jupyter adds them for you
after a colon.

```python
for r in [0.3, 1.0]:
    explore(resolution=r)     # inside the loop, runs twice
print("done")                 # outside, runs once
```

> **Coming from R?**
>
> ```r
> for (r in c(0.3, 0.6, 1.0)) {      # parentheses around the header
>   explore(resolution = r)          # braces mark the body
> }
> ```
>
> Python drops the parentheses and the braces, and uses a colon plus indentation
> instead. R's `c(...)` becomes `[...]`. The idea is identical.
>
> Two errors you will meet: forgetting the colon gives `SyntaxError`, and inconsistent
> indentation gives `IndentationError`. Both point at the right line.

In [ ]:
for r in [0.3, 0.6, 1.0, 2.0]:        # <-- CHANGE THIS list
    explore(resolution=r, show=False)

### Now write one yourself

Above you ran `explore(n_neighbors=5)` once. Write a loop that tries several values
instead.

Fill in the two blanks in the next cell:

- the **list** of values to try — pick three or four between 5 and 50
- the **argument** inside `explore(...)`, so that each value is passed as `n_neighbors`

Remember the colon and the indent. If it errors, read the last line of the message
first.

In [ ]:
# Your loop. Try to make a for loop fot the number of neighbors

**Once that runs**, try two extensions:

```python
# loop over something that is not numbers
for name in ["leiden", "total_counts", "cell_area"]:
    print(name)

# two parameters at once — a loop inside a loop
for n in [10, 30]:
    for r in [0.5, 1.5]:
        explore(n_neighbors=n, resolution=r, show=False)
```

The nested version runs four times: every `r` for every `n`. Watch the indentation —
the inner loop is inside the outer one, so it is indented twice.

Careful with nesting, though. Each combination is a full re-clustering, so a loop of
5 × 5 is 25 runs and a coffee break.

> **Try it yourself — the UMAP itself**
>
> These two change only the *picture*, never the clustering. `min_dist` sets how
> tightly points may pack; `spread` sets the overall scale. Run both and notice that
> the cluster count printed above does not move — good evidence for the claim on the
> slides that UMAP is a drawing, not an analysis.

In [ ]:
explore(min_dist=0.05, spread=1.0)     # <-- tight
explore(min_dist=0.8, spread=2.0)      # <-- loose

### Now commit

Put your chosen values in the cell below. It runs the same pipeline on **all** the
cells, and everything after this point uses the result.

If you are unsure, the defaults are reasonable. You can come back and change them
after you have seen the markers in section 3 — that is the normal way round, not a
failure of planning.

In [ ]:
# ---- YOUR CHOICE ---------------------------------------------------------
N_PCS = 30            # <-- CHANGE THIS
N_NEIGHBORS = 15      # <-- CHANGE THIS
RESOLUTION = 1.0      # <-- CHANGE THIS
MIN_DIST = 0.5        # <-- CHANGE THIS (picture only)
# --------------------------------------------------------------------------

sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
sc.tl.umap(adata, min_dist=MIN_DIST)
sc.tl.leiden(adata, resolution=RESOLUTION, key_added="leiden",
             flavor="igraph", n_iterations=2)

adata.uns["clustering_params"] = {
    "n_pcs": N_PCS, "n_neighbors": N_NEIGHBORS,
    "resolution": RESOLUTION, "min_dist": MIN_DIST,
    "n_clusters": int(adata.obs["leiden"].nunique()),
}
print(adata.obs["leiden"].value_counts().sort_index())
print(f"\n{adata.obs['leiden'].nunique()} clusters, recorded in adata.uns")

In [ ]:
sc.pl.umap(adata, color=["leiden", "total_counts", "cell_area"],
           wspace=0.35, ncols=3, size=6)

**Check the second and third panels before you look at anything else.** If a cluster
lights up on `total_counts` or `cell_area` alone, it is a technical cluster — depth or
segmentation size — not a cell type. That happens far more often here than in
scRNA-seq, because the dynamic range of counts per cell is so compressed.

Four checks, in order of how quickly they settle it:

1. Is the cluster at an extreme of counts or area? → suspicious
2. Are its markers a coherent lineage, or a grab-bag? → grab-bag is bad
3. Where is it in the tissue? Scattered → artefact. A structure → probably real.
4. Does it survive a different normalisation? Appendix B has the tools.

## 3. Differentially expressed genes and annotation

`rank_genes_groups` compares each cluster against all the others and ranks genes by
how well they separate it. The result is a table of **differentially expressed genes
(DEGs)** per cluster, and it is what you use to work out what each cluster is.

With a targeted panel this reads unusually well, because every gene in the output was
deliberately chosen by somebody — there is no long tail of ribosomal and mitochondrial
genes to wade through.

> **One honest caveat about the p-values.** The clusters were defined from the same
> expression data the test now uses, so the test is not independent of the grouping.
> This inflates significance — every cluster will have "highly significant" DEGs, even
> clusters split from a single homogeneous population. It is a known problem
> (sometimes called double dipping or selective inference) and it affects every
> scRNA-seq paper you have read.
>
> In practice: use the **ranking and the effect sizes** to identify cell types, which
> is what we are doing. Do not quote the adjusted p-value as evidence that a cluster
> is real. If you need that, the honest test is whether the cluster reappears in an
> independent sample.

In [ ]:
# DEGs per cluster: Wilcoxon rank-sum, each cluster against all others.
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")

# The dotplot shows the top DEGs per cluster. Colour is mean expression within the
# cluster; dot size is the fraction of cells in that cluster expressing the gene.
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, standard_scale="var", cmap="Blues")

### Export the DEGs

The full table, for every cluster, in a form you can send to someone or open in Excel.

Columns worth knowing:

| Column | Meaning |
|---|---|
| `logfoldchanges` | log2 fold change, this cluster versus all others |
| `pct_in` / `pct_out` | % of cells expressing the gene inside / outside the cluster |
| `scores` | the Wilcoxon statistic used for the ranking |
| `pvals_adj` | Benjamini–Hochberg adjusted — read the caveat above before using it |
| `above_background` | from notebook 02: did this gene beat the negative-control null? |

That last column is the one to sort on before you believe a DEG you have not seen
before.

In [ ]:
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# Every cluster, every gene — sc.get pulls the whole rank_genes_groups result
# into a tidy table rather than the nested arrays scanpy stores internally.
deg = sc.get.rank_genes_groups_df(adata, group=None)
deg = deg.rename(columns={"group": "cluster"})

# fraction of cells expressing each gene, inside and outside its cluster
counts = adata.layers["counts"]
expressed = (counts > 0)
for cluster in adata.obs["leiden"].cat.categories:
    m = (adata.obs["leiden"] == cluster).to_numpy()
    inside = np.asarray(expressed[m].mean(axis=0)).ravel() * 100
    outside = np.asarray(expressed[~m].mean(axis=0)).ravel() * 100
    pct = pd.DataFrame({"names": adata.var_names, "pct_in": inside, "pct_out": outside})
    rows = deg["cluster"] == cluster
    deg.loc[rows, ["pct_in", "pct_out"]] = (
        deg.loc[rows, ["names"]].merge(pct, on="names", how="left")
        [["pct_in", "pct_out"]].to_numpy())

# carry the notebook 02 detection flag across, if it is there
if "above_background" in adata.var.columns:
    deg["above_background"] = deg["names"].map(adata.var["above_background"])

# label with your annotation as well as the cluster number, when it exists
if "cell_type" in adata.obs.columns:
    lookup = (adata.obs[["leiden", "cell_type"]].astype(str)
              .drop_duplicates().set_index("leiden")["cell_type"])
    deg.insert(1, "cell_type", deg["cluster"].map(lookup))

deg = deg.round(4)
print(f"{len(deg):,} rows — {deg['cluster'].nunique()} clusters x {adata.n_vars:,} genes")
display(deg.head(10))

In [ ]:
# Write it out. CSV always works; xlsx is nicer to hand to someone, with one
# sheet per cluster.
csv_path = RESULTS / "DEGs_all_clusters.csv"
deg.to_csv(csv_path, index=False)
print(f"wrote {csv_path.relative_to(ROOT)}  ({csv_path.stat().st_size / 1e6:.1f} MB)")

# A trimmed version is usually the one people actually read.
TOP_N = 25              # <-- CHANGE THIS
MIN_LOGFC = 0.5         # <-- CHANGE THIS

top = (deg[(deg["logfoldchanges"] >= MIN_LOGFC) & (deg["pvals_adj"] < 0.05)]
       .sort_values(["cluster", "scores"], ascending=[True, False])
       .groupby("cluster", observed=True)
       .head(TOP_N))
top_path = RESULTS / f"DEGs_top{TOP_N}_per_cluster.csv"
top.to_csv(top_path, index=False)
print(f"wrote {top_path.relative_to(ROOT)}  ({len(top):,} rows)")

def sheet_name(label):
    """Excel forbids / \\ ? * [ ] in sheet names and caps them at 31 characters."""
    clean = "".join("-" if ch in r"/\\?*[]:" else ch for ch in str(label))
    return clean[:31] or "sheet"


try:
    xlsx_path = RESULTS / "DEGs_per_cluster.xlsx"
    with pd.ExcelWriter(xlsx_path) as writer:
        top.to_excel(writer, sheet_name="all_clusters", index=False)
        for cluster, block in top.groupby("cluster", observed=True):
            block.to_excel(writer, sheet_name=sheet_name(cluster), index=False)
    print(f"wrote {xlsx_path.relative_to(ROOT)}  "
          f"({top['cluster'].nunique() + 1} sheets)")
except ImportError:
    print("openpyxl not installed, so no Excel file — the CSVs above have the same")
    print("content. Install it with:  pip install openpyxl")

### Before you send that file to anyone

Two sanity checks, both quick.

1. **Sort by `above_background`.** A DEG that never rose above the negative-control
   null in notebook 02 is not a finding, whatever its p-value says.
2. **Look at `pct_in` versus `pct_out`.** A gene expressed in 12% of the cluster and
   8% of everything else can still come out "significant" with enough cells, and it is
   not a marker. You want a large gap, not a small p-value.

If you want a single number to sort on, `logfoldchanges` combined with a `pct_in`
threshold serves better than significance for this kind of data.

### A marker set for this dataset

These are taken from `rank_genes_groups` run on **this** ovarian section, not from a
textbook list — which is why several classic markers are absent. Two things worth
noticing before you use them:

- **`PAX8`, `WT1` and `MUC16` are not among the top discriminating genes here.** The
  tumour clusters separate instead on hypoxia- and growth-associated genes (`NDRG1`,
  `TFPI2`, `H19`, `LAPTM4B`). Atlas markers do not always transfer to a targeted
  panel in a specific tumour.
- **There is a ciliated epithelial population** (`FAM183A`, `CFAP100`) that a generic
  ovarian marker list would miss entirely. In tubo-ovarian tissue that is expected and
  anatomically meaningful — you should be able to say where those cells are.

Treat this as a scaffold. Your cluster numbers will differ, and so may the biology if
you use a different crop.

If a marker you expected is missing, check `gene_panel.json` from the run before
concluding anything — the matrix is not the panel. Appendix A has more on why a
workhorse gene can be absent.

In [ ]:
def find_gene(adata, name):
    """Exact match, then case-insensitive, then report near-misses."""
    if name in adata.var_names:
        return name
    lower = {g.lower(): g for g in adata.var_names}
    if name.lower() in lower:
        return lower[name.lower()]
    return None


def build_signature(adata, primary, alternates=()):
    """Take the genes that are on the panel; fall back to alternates if short.

    Written this way because marker lists do not survive contact with a targeted
    panel. A signature is a claim about a cell type, not about specific genes —
    if COL1A1 is absent, another fibrillar collagen makes the same claim.
    """
    found = [g for g in (find_gene(adata, n) for n in primary) if g]
    missing = [n for n in primary if find_gene(adata, n) is None]
    used_alt = []
    if len(found) < 3:
        for n in alternates:
            g = find_gene(adata, n)
            if g and g not in found:
                found.append(g)
                used_alt.append(g)
            if len(found) >= 4:
                break
    return found, missing, used_alt


# Primary markers, plus alternates that make the same biological claim. The
# alternates are only used when too few primaries survive.
markers = {
    "Tumour / epithelial": (
        ["EPCAM", "CP", "LAPTM4B", "NDRG1", "TFPI2", "H19", "UCHL1", "PLXNB1", "CD47"],
        ["KRT8", "KRT18", "KRT19", "MUC1", "CLDN4"]),
    "Tumour, proliferating": (
        ["TOP2A", "BIRC5", "SMC4", "HNRNPD"],
        ["MKI67", "PCNA", "CCNB1", "CDK1", "UBE2C"]),
    "Ciliated epithelium": (
        ["FAM183A", "CFAP100"],
        ["FOXJ1", "PIFO", "TPPP3", "CAPS", "DNAI1"]),
    "Fibroblast / CAF": (
        ["DCN", "LUM", "POSTN", "BGN", "C7", "OGN"],
        ["ASPN", "FMOD", "MGP", "SPARC", "FBLN1"]),
    "Fibroblast, adventitial": (
        ["PI16", "MFAP5", "TIMP3"],
        ["SCARA5", "CD34", "PCOLCE2"]),
    "Fibroblast, matrix-high": (
        ["COL1A1", "COL1A2", "COL4A1", "COL4A2"],
        ["COL3A1", "COL5A1", "COL6A1", "COL6A2", "COL6A3", "FN1"]),
    "Smooth muscle": (
        ["MYH11", "MYL9", "TAGLN", "C11orf96"],
        ["ACTA2", "CNN1", "DES", "ACTG2"]),
    "Endothelial": (
        ["FLT1", "EPAS1", "AQP1", "PECAM1", "SOCS3", "TFPI"],
        ["VWF", "CDH5", "CLDN5", "RAMP2", "EGFL7"]),
    "T cell": (
        ["TRAC", "TRBC1", "CD52", "CXCR4"],
        ["CD3D", "CD3E", "CD2", "IL7R", "CD8A"]),
    "Myeloid / macrophage": (
        ["C1QC", "MS4A6A", "AIF1", "FCGR3A", "FCGBP"],
        ["CD68", "CD14", "C1QA", "C1QB", "LYZ", "TYROBP"]),
}

present, report = {}, []
for name, (primary, alternates) in markers.items():
    found, missing, used_alt = build_signature(adata, primary, alternates)
    if found:
        present[name] = found
    report.append({
        "signature": name,
        "n used": len(found),
        "missing from panel": ", ".join(missing) if missing else "",
        "alternates pulled in": ", ".join(used_alt) if used_alt else "",
    })
display(pd.DataFrame(report).set_index("signature"))

thin = [r["signature"] for r in report if r["n used"] < 2]
if thin:
    print(f"Too few genes to score reliably: {thin}")
    print("Add alternates by hand, or drop the signature.")

In [ ]:
sc.pl.dotplot(adata, present, groupby="leiden", standard_scale="var",
              cmap="Blues", figsize=(14, 5))

> **Try it yourself — where is one gene expressed?**
>
> Marker tables are abstract. Put a single gene straight onto the tissue and it
> becomes obvious. Try a marker you know well, then one you do not.

In [ ]:
GENE = "EPCAM"        # <-- CHANGE THIS to any gene on the panel

if GENE not in adata.var_names:
    close = [g for g in adata.var_names if GENE.upper() in g.upper()][:10]
    print(f"{GENE!r} is not on the panel. Did you mean: {close}")
else:
    expr = np.asarray(adata[:, GENE].X.todense()).ravel()
    order = np.argsort(expr)             # draw high-expressing cells last, on top
    xs, ys = adata.obsm["spatial"].T

    fig, ax = plt.subplots(figsize=(6, 6))
    pc = ax.scatter(xs[order], ys[order], c=expr[order], s=1.4, cmap="magma",
                    vmax=np.percentile(expr, 99), linewidths=0, rasterized=True)
    plt.colorbar(pc, ax=ax, fraction=0.046, label=f"{GENE} (log-normalised)")
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(GENE)
    plt.show()

    pct = 100 * (expr > 0).mean()
    print(f"{GENE} detected in {pct:.1f}% of cells")

In [ ]:
# score each cell for each signature — more robust than eyeballing single genes
for name, genes in present.items():
    if genes:
        sc.tl.score_genes(adata, genes, score_name=f"score_{name}")

score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
mean_scores = adata.obs.groupby("leiden", observed=True)[score_cols].mean()

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap((mean_scores - mean_scores.mean()) / mean_scores.std(),
            cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "z-scored mean signature score"})
ax.set_xticklabels([c.replace("score_", "") for c in score_cols], rotation=40, ha="right")
plt.tight_layout(); plt.show()

### Now assign labels

Edit the dictionary below to match **your** clusters — the numbers will not be the
same as anyone else's, because Leiden is not deterministic across versions.

In [ ]:
# Automatic first pass: give each cluster the signature it scores highest on.
# Treat it as a draft. You are the anatomist; overwrite anything that looks wrong.
auto = mean_scores.idxmax(axis=1).str.replace("score_", "", regex=False)
print(auto.to_string())

annotation = auto.to_dict()
# --- override by hand, e.g.: ---
# annotation["7"] = "Tumour (proliferating)"
# annotation["11"] = "Doublet / segmentation artefact"

adata.obs["cell_type"] = adata.obs["leiden"].map(annotation).astype("category")
adata.obs["cell_type"].value_counts()

## 4. The cluster that is not a cell type

Ask of every cluster: *could this be two cells in one polygon?*

A cluster co-expressing markers of two lineages that are physically adjacent —
tumour + immune, endothelium + fibroblast — is the classic spatial artefact. In
scRNA-seq you would call it a doublet and move on. Here you can **go and look at it**.

In [ ]:
# Pairs of lineages that are physically adjacent in this tissue but should not
# be co-expressed in one cell. Names must match the keys in `present` above.
lineage_pairs = [("Tumour / epithelial", "T cell"),
                 ("Tumour / epithelial", "Myeloid / macrophage"),
                 ("Endothelial", "Fibroblast / CAF")]

missing = {n for pair in lineage_pairs for n in pair
           if f"score_{n}" not in adata.obs.columns}
if missing:
    print(f"no score column for {sorted(missing)} — those pairs will be skipped")
    print(f"available: {[c[6:] for c in adata.obs.columns if c.startswith('score_')]}")

rows = []
for a, b in lineage_pairs:
    if f"score_{a}" in adata.obs and f"score_{b}" in adata.obs:
        za = (adata.obs[f"score_{a}"] - adata.obs[f"score_{a}"].mean()) / adata.obs[f"score_{a}"].std()
        zb = (adata.obs[f"score_{b}"] - adata.obs[f"score_{b}"].mean()) / adata.obs[f"score_{b}"].std()
        both = ((za > 1) & (zb > 1))
        adata.obs[f"mixed_{a[:4]}_{b[:4]}"] = both
        rows.append((f"{a} + {b}", int(both.sum()), 100 * both.mean()))
pd.DataFrame(rows, columns=["pair", "n cells", "% of cells"]).round(2)

In [ ]:
col = [c for c in adata.obs.columns if c.startswith("mixed_")][0]
fig, ax = plt.subplots(figsize=(6, 6))
x, y = adata.obsm["spatial"].T
m = adata.obs[col].to_numpy()
ax.scatter(x, y, s=0.8, c="0.85", linewidths=0, rasterized=True)
ax.scatter(x[m], y[m], s=4, c=CORAL, linewidths=0, rasterized=True)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"cells scoring high for both lineages ({col})")
plt.show()

**Where are they?** If double-positive cells sit exactly on the interface between two
tissue compartments, they are boundary segmentation errors, not a hybrid cell state.
If they are scattered at random, the cause is more likely diffuse background.
If they form a compact focus of their own — think again, that could be real
(e.g. tumour cells that have engulfed something, or a genuine mixed niche).

This diagnosis is *only* available because you know where the cells are.

## 5. The payoff: put the cell types back on the tissue

In [ ]:
def spatial_types(adata, key="cell_type", s=1.4, figsize=(9, 8), title=None):
    """Cell types on the tissue, using adata.uns[f"{key}_colors"] if it is set.

    Reading the palette from the object rather than hard-coding one means the
    colours you choose below apply here and in scanpy's own plots, and travel
    with the object into notebooks 04 and 05.
    """
    cats = adata.obs[key].astype("category")
    colours = adata.uns.get(f"{key}_colors")
    if colours is None or len(colours) != len(cats.cat.categories):
        cm = plt.get_cmap("tab20")
        colours = [cm(i % 20) for i in range(len(cats.cat.categories))]

    fig, ax = plt.subplots(figsize=figsize)
    x, y = adata.obsm["spatial"].T
    for colour, name in zip(colours, cats.cat.categories):
        m = (cats == name).to_numpy()
        ax.scatter(x[m], y[m], s=s, color=colour, label=f"{name} ({m.sum():,})",
                   linewidths=0, rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False,
              markerscale=8, fontsize=9)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    return ax


spatial_types(adata)
plt.show()

Compare this with the UMAP. The UMAP tells you which cells are *similar*. The tissue
plot tells you which cells are *together*. Those are different facts, and only one of
them was measured.

Look for: tumour nests with sharp edges, stromal bands, vessels as thin lines, immune
cells at the interface or excluded from it. If your clustering is right, the plot
should look like histology. **If it looks like static, your clustering is wrong** —
and that is a validation route scRNA-seq simply does not have.

> **Try it yourself — choose your own colours**
>
> The default palette gives every cluster an equally loud colour, which is the worst
> possible choice for reading a tissue map: everything competes and nothing stands out.
> Choosing colours is a real analytical decision, not decoration.
>
> Three principles that carry most of the benefit:
>
> - **Grey is a colour.** Push populations you are not currently interested in to grey
>   or a pale tint. One or two saturated colours against grey reads instantly.
> - **Group by lineage.** Give the three fibroblast states three shades of one hue, and
>   the immune populations a different hue. Then the eye groups them for you.
> - **Avoid red-green pairs** for the contrast that carries your argument — roughly one
>   man in twelve will not see it.
>
> Set your colours below. Because we store them in `adata.uns["cell_type_colors"]`,
> scanpy picks them up automatically — every `sc.pl.umap(color="cell_type")` from here
> on uses them, and so does the tissue plot.

In [ ]:
# Your palette. Every category needs an entry; anything you leave out falls back
# to light grey, which is a perfectly good way to say "not the point right now".
MY_COLOURS = {
    "Tumour / epithelial":      "#001158",   # navy   <-- CHANGE THESE
    "Tumour, proliferating":    "#3C6BC4",   # lighter navy — same lineage, same hue
    "Ciliated epithelium":      "#7FA8E8",
    "Fibroblast / CAF":         "#F26B43",   # orange family for stroma
    "Fibroblast, adventitial":  "#F8A07A",
    "Fibroblast, matrix-high":  "#B8431F",
    "Smooth muscle":            "#FBAE40",
    "Endothelial":              "#2E7D32",   # green for vessels
    "T cell":                   "#7B4B94",   # purple family for immune
    "Myeloid / macrophage":     "#B48EAD",
}
FALLBACK = "#DDDDDD"

cats = list(adata.obs["cell_type"].cat.categories)
palette = [MY_COLOURS.get(c, FALLBACK) for c in cats]
adata.uns["cell_type_colors"] = palette      # scanpy reads this automatically

missing = [c for c in cats if c not in MY_COLOURS]
unused = [c for c in MY_COLOURS if c not in cats]
print(f"{len(cats)} cell types; {len(cats) - len(missing)} have a colour")
if missing:
    print(f"  falling back to grey: {missing}")
if unused:
    print(f"  in your palette but not in the data (typo?): {unused}")

# a quick swatch, so you can see the palette before you commit to it
fig, ax = plt.subplots(figsize=(7, 0.45 * len(cats)))
for i, (c, col) in enumerate(zip(cats, palette)):
    n = int((adata.obs["cell_type"] == c).sum())
    ax.barh(i, 1, color=col)
    ax.text(1.03, i, f"{c}  ({n:,} cells)", va="center", fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(-0.6, len(cats) - 0.4)
ax.axis("off"); ax.invert_yaxis()
plt.show()

In [ ]:
# The same two plots as above, now with your palette. Judge them side by side.
sc.pl.umap(adata, color="cell_type", size=6, title="UMAP — your colours")
spatial_types(adata)
plt.show()

**Now ask whether it reads better.** Specifically:

- Can you find the tumour compartment in under a second?
- Do the three fibroblast states look related, or like three unrelated populations?
- Is anything important disappearing into grey that should not be?

Then try the version that makes a point: set **everything except one lineage** to grey
and re-run. That is the figure you would actually publish when the argument is about
one population — and it is much harder to misread than ten competing colours.

> Colours are stored in the object, so they travel with it into notebooks 04 and 05.
> If you later add or rename a cluster, re-run this cell, because scanpy matches the
> list to the category order rather than by name.

In [ ]:
# Try it: one lineage in colour, everything else grey.
FOCUS = "T cell"        # <-- CHANGE THIS to any cell type

focus_palette = ["#7B4B94" if c == FOCUS else "#E8E8E8"
                 for c in adata.obs["cell_type"].cat.categories]
adata.uns["cell_type_colors"] = focus_palette
spatial_types(adata)
plt.show()

# put your full palette back
adata.uns["cell_type_colors"] = palette

In [ ]:
# a couple of individual clusters on their own, which is far more readable
for ct in list(adata.obs["cell_type"].cat.categories)[:3]:
    m = (adata.obs["cell_type"] == ct).to_numpy()
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    x, y = adata.obsm["spatial"].T
    ax.scatter(x, y, s=0.5, c="0.88", linewidths=0, rasterized=True)
    ax.scatter(x[m], y[m], s=2.2, c=NAVY, linewidths=0, rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{ct}  (n={m.sum():,})")
    plt.show()

In [ ]:
import shutil

dest = DATA / "ovarian_annotated.h5ad"
reference = DATA / "ovarian_annotated_reference.h5ad"

# The file you copied from the share has this same name. Keep a pristine copy
# the first time you overwrite it, so you can always fall back to the shipped
# annotation without re-copying from P:.
if dest.exists() and not reference.exists():
    shutil.copy2(dest, reference)
    print(f"kept the shipped version as {reference.name}")

adata.write_h5ad(dest, compression="gzip")
print(f"saved -> {dest.name}  ({dest.stat().st_size / 1e6:.0f} MB)")
print("\nDay 2 loads this file, so notebooks 04-06 will use YOUR annotation.")
print(f"To go back to the shipped one, copy {reference.name} over it.")

### Exercise 3.1 — does your clustering survive its parameters?

Pick the two settings you were least sure about in section 2 and re-run `explore()`
with each. Then cross-tabulate the labelings:

```python
pd.crosstab(sub.obs["cl"], other_labels)
```

Which cell types stay together whatever you do, and which split and merge? A
population that only exists at one resolution is a claim you should not make.

### Exercise 3.2
Take your smallest cluster. Is it a rare cell type, a technical artefact, or a
region-specific state? Give three pieces of evidence, at least one of which is spatial.

### Exercise 3.3
Re-run the cell-area profile idea from section 2 using a **marker** gene rather than
an abundant one — `TRAC` or `C1QC`. Immune cells are small, so anything that depends
on cell size will show up more strongly here than for a ubiquitous gene.

> Appendix B (`A2_normalisation_and_depth.ipynb`) has the harder version of exercise
> 3.1: re-cluster under four different normalisations and decide which to believe,
> using the tissue as the arbiter.

---
**Next:** `04_spatial_statistics.ipynb` — the part scRNA-seq cannot do at all.